# components.table

> Table layout rendering: header row, data rows, and cells using CSS table display.

In [ ]:
#| default_exp components.table

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional, Tuple

from fasthtml.common import Div, Span

from cjm_fasthtml_tailwind.utilities.spacing import p
from cjm_fasthtml_tailwind.utilities.typography import font_weight, truncate, font_size
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items as items_align, gap
from cjm_fasthtml_tailwind.utilities.interactivity import cursor as cursor_tw
from cjm_fasthtml_tailwind.utilities.layout import display_tw
from cjm_fasthtml_tailwind.utilities.tables import table_display
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_daisyui.utilities.semantic_colors import border_dui, bg_dui, text_dui

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V11 icon-size roles)
from cjm_fasthtml_design_system.icons import icons

from cjm_fasthtml_virtual_collection.core.models import (
    ColumnDef, VirtualCollectionConfig, VirtualCollectionState,
    VirtualCollectionUrls, CellRenderContext, RowRenderContext,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

## Header Row

In [ ]:
#| export
SORT_ICON_SIZE = icons.dense_inline  # Tailwind size scale for sort indicator icons (V11 dense_inline role)

def _sort_indicator(column: ColumnDef,  # Column definition
                    state: VirtualCollectionState,  # Collection state (for current sort)
                   ) -> Any:  # Sort icon element or empty string
    """Render sort indicator icon for a sortable header cell."""
    if not column.sortable:
        return ""
    if state.sort_column == column.key:
        icon_name = "arrow-up" if state.sort_ascending else "arrow-down"
        return lucide_icon(icon_name, size=SORT_ICON_SIZE, cls=str(text_dui.primary))
    return lucide_icon("arrow-up-down", size=SORT_ICON_SIZE,
                       cls=str(text_dui.base_content))


def render_header_cell(column: ColumnDef,  # Column definition
                       state: VirtualCollectionState,  # Collection state
                       sort_url: str = "",  # Sort URL (empty = no click-to-sort)
                      ) -> Div:  # Header cell element
    """Render a single table header cell with optional sort indicator."""
    indicator = _sort_indicator(column, state)

    if column.header and indicator:
        # Flex container for text + icon inline
        content = Div(
            Span(column.header),
            indicator,
            cls=combine_classes(flex_display, items_align.center, gap(1)),
        )
    elif column.header:
        content = Span(column.header)
    else:
        content = ""

    cls = combine_classes(
        table_display.cell,
        p.x(2), p.y(1),
        font_weight.semibold, font_size.sm,
        border.b(), border_dui.base_300,
        cursor_tw.pointer if column.sortable and sort_url else None,
        column.header_cls or None,
    )

    # Click-to-sort HTMX attributes
    sort_attrs = {}
    if column.sortable and sort_url:
        sort_attrs = dict(
            hx_post=f"{sort_url}?column={column.key}",
            hx_swap="none",
        )

    return Div(content, cls=cls, **sort_attrs)

In [ ]:
#| export
def render_header_row(config: VirtualCollectionConfig,  # Collection config
                      ids: VirtualCollectionHtmlIds,     # HTML IDs
                      state: VirtualCollectionState = None,  # Collection state (for sort indicators)
                      sort_url: str = "",                # Sort URL (empty = no click-to-sort)
                     ) -> Div:  # Header row element
    """Render the table header row with optional sort indicators."""
    if state is None:
        state = VirtualCollectionState()
    cells = [render_header_cell(col, state, sort_url) for col in config.columns]
    return Div(
        *cells,
        id=ids.header,
        cls=combine_classes(table_display.row, bg_dui.base_200),
    )

## Data Rows

In [ ]:
#| export
def render_data_cell(item: Any,                    # Data item
                     column: ColumnDef,             # Column definition
                     row_index: int,                # Row index
                     total_items: int,              # Total item count
                     ids: VirtualCollectionHtmlIds,  # HTML IDs
                     render_cell: Callable,          # Consumer cell render callback
                     is_cursor: bool = False,        # Whether row is keyboard cursor
                    ) -> Div:  # Cell element with stable ID
    """Render a single data cell with a stable ID for OOB updates."""
    ctx = CellRenderContext(
        row_index=row_index,
        column=column,
        total_items=total_items,
        is_cursor=is_cursor,
    )
    content = render_cell(item, ctx)
    cls = combine_classes(
        table_display.cell,
        p.x(2), p.y(1),
        border.b(), border_dui.base_200,
        column.cell_cls or None,
    )
    return Div(
        content,
        id=ids.cell_id(row_index, column.key),
        cls=cls,
    )

In [ ]:
#| export
def render_data_row(item: Any,                       # Data item
                    row_index: int,                   # Row index in full collection
                    config: VirtualCollectionConfig,   # Collection config
                    state: VirtualCollectionState,     # Collection state
                    ids: VirtualCollectionHtmlIds,     # HTML IDs
                    render_cell: Callable,             # Consumer cell render callback
                    focus_url: str = "",               # URL for click-to-focus (empty = disabled)
                   ) -> Div:  # Row element with stable ID
    """Render a single data row with all cells."""
    is_cursor = (row_index == state.cursor_index)

    cells = [
        render_data_cell(
            item, col, row_index, state.total_items, ids, render_cell, is_cursor
        )
        for col in config.columns
    ]

    row_cls = combine_classes(
        table_display.row,
        bg_dui.base_200.hover,
        cursor_tw.pointer if focus_url else None,
        bg_dui.base_300 if is_cursor else None,
    )

    # Click-to-focus HTMX attributes (skip clicks on interactive children)
    focus_attrs = {}
    if focus_url:
        focus_attrs = dict(
            hx_post=f"{focus_url}?row_index={row_index}",
            hx_swap="none",
            hx_trigger="click[!event.target.closest('input,button,a')]",
        )

    return Div(
        *cells,
        id=ids.row_id(row_index),
        cls=row_cls,
        **focus_attrs,
    )

## Slots

In [ ]:
#| export
def render_slot(
    slot_index: int,                       # Position in viewport (0-based)
    item: Any,                             # Data item
    item_index: int,                       # Row index in full collection
    config: VirtualCollectionConfig,       # Collection config
    state: VirtualCollectionState,         # Collection state
    ids: VirtualCollectionHtmlIds,         # HTML IDs
    render_cell: Callable,                 # Consumer cell render callback
    oob: bool = False,                     # Whether to render as OOB swap
    focus_url: str = "",                   # URL for click-to-focus (empty = disabled)
) -> Div:  # Slot wrapper (display:contents) containing the data row
    """Render a viewport slot wrapping a data row with display:contents."""
    inner_row = render_data_row(
        item, item_index, config, state, ids, render_cell, focus_url=focus_url,
    )
    return Div(
        inner_row,
        id=ids.slot_id(slot_index),
        cls=combine_classes(display_tw.contents),
        hx_swap_oob="innerHTML" if oob else None,
    )

In [ ]:
#| export
def render_table_rows(items: list,                       # Full item list
                      config: VirtualCollectionConfig,    # Collection config
                      state: VirtualCollectionState,      # Collection state
                      ids: VirtualCollectionHtmlIds,      # HTML IDs
                      render_cell: Callable,              # Consumer cell render callback
                      focus_url: str = "",                # URL for click-to-focus (empty = disabled)
                     ) -> Div:  # Table-row-group container with slot wrappers
    """Render all visible rows in the current window as a table-row-group."""
    from cjm_fasthtml_virtual_collection.core.windowing import compute_window

    render_start, render_end = compute_window(
        state.window_start, state.visible_rows, state.total_items
    )

    slots = [
        render_slot(
            slot_index=slot_idx, item=items[item_idx], item_index=item_idx,
            config=config, state=state, ids=ids, render_cell=render_cell,
            focus_url=focus_url,
        )
        for slot_idx, item_idx in enumerate(range(render_start, render_end))
    ]

    return Div(*slots, id=ids.rows, cls=combine_classes(table_display.row_group))

## Tests

## OOB Helpers

In [ ]:
#| export
def render_cell_oob(item: Any,                       # Data item
                    column: ColumnDef,                # Column to render
                    row_index: int,                   # Row index
                    total_items: int,                 # Total item count
                    ids: VirtualCollectionHtmlIds,    # HTML IDs
                    render_cell: Callable,            # Consumer cell render callback
                    is_cursor: bool = False,          # Whether row is keyboard cursor
                   ) -> Div:  # Cell element with hx-swap-oob
    """Render a single cell with OOB swap for targeted update."""
    cell = render_data_cell(item, column, row_index, total_items, ids, render_cell, is_cursor)
    cell.attrs["hx-swap-oob"] = "outerHTML"
    return cell


def render_row_oob(item: Any,                       # Data item
                   row_index: int,                   # Row index
                   config: VirtualCollectionConfig,  # Collection config
                   state: VirtualCollectionState,    # Collection state
                   ids: VirtualCollectionHtmlIds,    # HTML IDs
                   render_cell: Callable,            # Consumer cell render callback
                  ) -> Div:  # Row element with hx-swap-oob
    """Render a full row with OOB swap for targeted update."""
    row = render_data_row(item, row_index, config, state, ids, render_cell)
    row.attrs["hx-swap-oob"] = "outerHTML"
    return row


def render_visible_cells_oob(
    column: ColumnDef,                # Column to re-render
    item_indices: List[int],          # Item indices that changed
    items: list,                      # Full items list
    state: VirtualCollectionState,    # Current VC state (for window bounds)
    ids: VirtualCollectionHtmlIds,    # HTML IDs
    render_cell: Callable,            # Consumer cell render callback
) -> Tuple[Div, ...]:                # OOB cell elements for visible items only
    """Batch-render OOB cell updates for specified items within the visible window."""
    from cjm_fasthtml_virtual_collection.core.windowing import compute_window

    render_start, render_end = compute_window(
        state.window_start, state.visible_rows, state.total_items
    )
    oob_cells = []
    for idx in item_indices:
        if render_start <= idx < render_end:
            oob_cells.append(render_cell_oob(
                items[idx], column, idx, state.total_items,
                ids, render_cell, is_cursor=(idx == state.cursor_index),
            ))
    return tuple(oob_cells)

In [ ]:
from fasthtml.common import to_xml
from cjm_fasthtml_virtual_collection.core.models import ColumnDef, VirtualCollectionConfig, VirtualCollectionState
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

# Test setup
columns = (
    ColumnDef(key="name", header="Name"),
    ColumnDef(key="size", header="Size"),
)
config = VirtualCollectionConfig(prefix="t", columns=columns)
state = VirtualCollectionState(total_items=5, visible_rows=3)
ids = VirtualCollectionHtmlIds(prefix=config.prefix)

In [ ]:
# Test render_header_row — uses table display classes
header = render_header_row(config, ids)
header_html = to_xml(header)
assert 'id="t-header"' in header_html
assert 'table-row' in header_html
assert 'table-cell' in header_html
assert 'Name' in header_html
assert 'Size' in header_html
print("Header row test passed")

In [ ]:
# Test render_data_row — uses table display classes
from dataclasses import dataclass

@dataclass
class Item:
    name: str
    size: str

test_items = [Item(f"file_{i}.txt", f"{i}KB") for i in range(5)]

def test_render_cell(item, ctx):
    if ctx.column.key == "name": return Span(item.name)
    if ctx.column.key == "size": return Span(item.size)

row = render_data_row(test_items[2], 2, config, state, ids, test_render_cell)
row_html = to_xml(row)
assert 'id="t-row-2"' in row_html
assert 'id="t-row-2-col-name"' in row_html
assert 'id="t-row-2-col-size"' in row_html
assert 'file_2.txt' in row_html
assert 'table-row' in row_html
assert 'table-cell' in row_html
assert 'style=' not in row_html
print("Data row test passed")

In [ ]:
# Test render_table_rows — slots with display:contents wrap rows in table-row-group
rows_container = render_table_rows(test_items, config, state, ids, test_render_cell)
rows_html = to_xml(rows_container)
assert 'id="t-rows"' in rows_html
assert 'table-row-group' in rows_html
# Slot wrappers with display:contents
assert 'id="t-slot-0"' in rows_html
assert 'id="t-slot-1"' in rows_html
assert 'id="t-slot-2"' in rows_html
assert 'id="t-slot-3"' not in rows_html  # visible_rows=3
assert 'contents' in rows_html  # display:contents on slots
# Data rows inside slots
assert 'id="t-row-0"' in rows_html
assert 'id="t-row-1"' in rows_html
assert 'id="t-row-2"' in rows_html
assert 'id="t-row-3"' not in rows_html
print("Table rows (with display:contents slots) test passed")

In [ ]:
# Test render_cell_oob
col_name = columns[0]  # "name" column
cell_oob = render_cell_oob(test_items[1], col_name, 1, 5, ids, test_render_cell)
cell_html = to_xml(cell_oob)
assert 'id="t-row-1-col-name"' in cell_html
assert 'hx-swap-oob="outerHTML"' in cell_html
assert 'file_1.txt' in cell_html
print("render_cell_oob test passed")

# Test render_row_oob
row_oob = render_row_oob(test_items[3], 3, config, state, ids, test_render_cell)
row_html = to_xml(row_oob)
assert 'id="t-row-3"' in row_html
assert 'hx-swap-oob="outerHTML"' in row_html
assert 'file_3.txt' in row_html
assert 'id="t-row-3-col-name"' in row_html
assert 'id="t-row-3-col-size"' in row_html
print("render_row_oob test passed")

In [ ]:
# Test render_visible_cells_oob — only visible items get OOB cells
# State: window_start=0, visible_rows=3, total_items=5 → visible indices 0,1,2
oob_cells = render_visible_cells_oob(col_name, [0, 2, 4], test_items, state, ids, test_render_cell)
assert len(oob_cells) == 2, f"Expected 2 visible cells, got {len(oob_cells)}"
# Index 0 and 2 are visible; index 4 is outside window
html_0 = to_xml(oob_cells[0])
html_2 = to_xml(oob_cells[1])
assert 'id="t-row-0-col-name"' in html_0
assert 'id="t-row-2-col-name"' in html_2
assert 'hx-swap-oob="outerHTML"' in html_0
assert 'hx-swap-oob="outerHTML"' in html_2
assert 'file_0.txt' in html_0
assert 'file_2.txt' in html_2
print("render_visible_cells_oob — visibility filter test passed")

In [ ]:
# Test render_visible_cells_oob — empty indices returns empty tuple
oob_empty = render_visible_cells_oob(col_name, [], test_items, state, ids, test_render_cell)
assert oob_empty == ()
print("render_visible_cells_oob — empty indices test passed")

# Test render_visible_cells_oob — all indices outside window returns empty tuple
oob_outside = render_visible_cells_oob(col_name, [3, 4], test_items, state, ids, test_render_cell)
assert oob_outside == ()
print("render_visible_cells_oob — all outside window test passed")

In [ ]:
# Test render_visible_cells_oob — cursor item gets is_cursor=True styling
state_with_cursor = VirtualCollectionState(total_items=5, visible_rows=3, cursor_index=1)
oob_cursor = render_visible_cells_oob(col_name, [0, 1], test_items, state_with_cursor, ids, test_render_cell)
assert len(oob_cursor) == 2
# Item at index 1 is the cursor — should have bg-base-300 (cursor highlight not on cell, but is_cursor passed)
# Verify both cells have correct IDs
html_non_cursor = to_xml(oob_cursor[0])
html_cursor = to_xml(oob_cursor[1])
assert 'id="t-row-0-col-name"' in html_non_cursor
assert 'id="t-row-1-col-name"' in html_cursor
print("render_visible_cells_oob — cursor item test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()